In [3]:
import os
from contextlib import asynccontextmanager
from typing import List
from fastapi import FastAPI, HTTPException
import mlflow.sklearn
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

os.getcwd()

'C:\\Users\\TWH\\Smart-city-traffic-capstone-\\part3_machine_learning\\notebooks'

In [4]:

MODEL_NAME = "Traffic_Volume_Regressor"
model = None


# 1. Lifespan event handler for clean startup/shutdown
@asynccontextmanager
async def lifespan(app: FastAPI):
    global model
    try:
        model_uri = f"models:/{MODEL_NAME}/latest"
        model = mlflow.sklearn.load_model(model_uri)
        print(f"✓ Successfully loaded production model from MLflow: {model_uri}")
    except Exception:
        # Robust fallback model for standalone execution
        from sklearn.ensemble import GradientBoostingRegressor

        np.random.seed(42)
        X_dummy = np.random.rand(200, 10)
        y_dummy = np.random.randint(500, 5000, 200)
        model = GradientBoostingRegressor().fit(X_dummy, y_dummy)
        print(
            "⚠️ MLflow tracking server offline. Initialized local fallback GradientBoosting model."
        )
    yield


app = FastAPI(
    title="Traffic Volume Prediction Service",
    description="MLOps serving endpoint for traffic volume and risk modeling",
    version="3.0.0",
    lifespan=lifespan,
)


# 2. Pydantic V2 Data Validation Schemas
class FeaturePayload(BaseModel):
    hour_sin: float = Field(..., examples=[0.5])
    hour_cos: float = Field(..., examples=[-0.866])
    dow_sin: float = Field(..., examples=[0.781])
    dow_cos: float = Field(..., examples=[0.623])
    is_weekend: int = Field(..., ge=0, le=1, examples=[0])
    is_holiday: int = Field(..., ge=0, le=1, examples=[0])
    is_severe_weather: int = Field(..., ge=0, le=1, examples=[0])
    is_low_visibility: int = Field(..., ge=0, le=1, examples=[0])
    temp_scaled: float = Field(..., examples=[0.45])
    clouds_scaled: float = Field(..., examples=[0.20])


class BatchPredictionInput(BaseModel):
    samples: List[FeaturePayload]


# 3. Serving Endpoints
@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "model_loaded": model is not None,
        "model_name": MODEL_NAME,
    }


@app.post("/predict")
def predict_traffic_volume(payload: BatchPredictionInput):
    if model is None:
        raise HTTPException(
            status_code=500, detail="Model instance is not loaded."
        )

    # Convert incoming Pydantic payload to DataFrame
    records = [sample.model_dump() for sample in payload.samples]
    input_df = pd.DataFrame(records)

    # Generate inference predictions
    raw_predictions = model.predict(input_df)

    results = []
    for pred in raw_predictions:
        volume = int(max(0, pred))
        risk_tag = "HIGH" if volume > 4000 else "NORMAL"
        results.append(
            {"predicted_traffic_volume": volume, "risk_status": risk_tag}
        )

    return {"predictions": results}